# Préparation des features

Ce notebook présente la préparation des variables utilisées pour la modélisation du risque de défaut crédit.

Le dataset Home Credit est composé de plusieurs fichiers. La table principale contient les informations de demande de crédit, tandis que les tables secondaires décrivent l’historique bancaire et les anciens crédits des clients.

L’objectif de cette étape est de construire un dataset final exploitable par les modèles de machine learning.

## Structure du dataset Home Credit

Le dataset utilisé contient plusieurs tables reliées entre elles par des identifiants clients ou crédits.

Les fichiers principaux sont :

| Fichier | Rôle |
|---|---|
| `application_train.csv` | Table principale d'entraînement avec la cible `TARGET` |
| `application_test.csv` | Table principale de test sans la cible |
| `bureau.csv` | Historique des crédits déclarés auprès d'autres institutions |
| `bureau_balance.csv` | Historique mensuel des crédits du bureau |
| `previous_application.csv` | Anciennes demandes de crédit chez Home Credit |
| `POS_CASH_balance.csv` | Historique des crédits POS et cash |
| `installments_payments.csv` | Historique des paiements par échéance |
| `credit_card_balance.csv` | Historique des cartes de crédit |
| `HomeCredit_columns_description.csv` | Description des variables |

La clé principale utilisée pour fusionner les informations client est `SK_ID_CURR`.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

raw_files = [
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "POS_CASH_balance.csv",
    "installments_payments.csv",
    "credit_card_balance.csv",
    "HomeCredit_columns_description.csv",
]

for file_name in raw_files:
    file_path = RAW_DATA_DIR / file_name
    print(f"{file_name} : {'OK' if file_path.exists() else 'MANQUANT'}")

application_train.csv : OK
application_test.csv : OK
bureau.csv : OK
bureau_balance.csv : OK
previous_application.csv : OK
POS_CASH_balance.csv : OK
installments_payments.csv : OK
credit_card_balance.csv : OK
HomeCredit_columns_description.csv : OK


In [2]:
raw_summary = []


def read_csv_sample(file_path, nrows=1000):
    """
    Lit un CSV avec plusieurs encodages possibles.

    Certains fichiers Home Credit, notamment le fichier de description
    des colonnes, ne sont pas toujours encodés en UTF-8.
    """
    encodings_to_try = ["utf-8", "latin1", "cp1252"]

    for encoding in encodings_to_try:
        try:
            return pd.read_csv(file_path, nrows=nrows, encoding=encoding), encoding
        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        f"Impossible de lire le fichier avec les encodages testés : {encodings_to_try}",
    )


for file_name in raw_files:
    file_path = RAW_DATA_DIR / file_name

    if file_path.exists():
        df_sample, encoding_used = read_csv_sample(file_path, nrows=1000)

        raw_summary.append(
            {
                "file": file_name,
                "n_columns": df_sample.shape[1],
                "sample_rows_loaded": df_sample.shape[0],
                "has_SK_ID_CURR": "SK_ID_CURR" in df_sample.columns,
                "has_TARGET": "TARGET" in df_sample.columns,
                "encoding_used": encoding_used,
            }
        )
    else:
        raw_summary.append(
            {
                "file": file_name,
                "n_columns": None,
                "sample_rows_loaded": None,
                "has_SK_ID_CURR": False,
                "has_TARGET": False,
                "encoding_used": None,
            }
        )

raw_summary_df = pd.DataFrame(raw_summary)
raw_summary_df

,file,n_columns,sample_rows_loaded,has_SK_ID_CURR,has_TARGET,encoding_used
0,application_train.csv,122,1000,True,True,utf-8
1,application_test.csv,121,1000,True,False,utf-8
2,bureau.csv,17,1000,True,False,utf-8
3,bureau_balance.csv,3,1000,False,False,utf-8
4,previous_application.csv,37,1000,True,False,utf-8
5,POS_CASH_balance.csv,8,1000,True,False,utf-8
6,installments_payments.csv,8,1000,True,False,utf-8
7,credit_card_balance.csv,23,1000,True,False,utf-8
8,HomeCredit_columns_description.csv,5,219,False,False,latin1


## Principe de préparation des features

Les tables secondaires contiennent souvent plusieurs lignes pour un même client.

Par exemple, un client peut avoir :
- plusieurs anciens crédits ;
- plusieurs lignes mensuelles d’historique ;
- plusieurs échéances de paiement ;
- plusieurs opérations de carte bancaire.

Pour pouvoir entraîner un modèle, il faut obtenir une seule ligne par client.
Les tables secondaires sont donc agrégées par `SK_ID_CURR`.

Les agrégations utilisées sont principalement :
- moyenne ;
- somme ;
- minimum ;
- maximum ;
- nombre d’observations ;
- écart-type.

Ces agrégations permettent de transformer l’historique client en variables numériques exploitables par les modèles.

## Fichiers intermédiaires générés

Après agrégation des tables secondaires, plusieurs fichiers intermédiaires sont créés dans `data/interim/`.

| Fichier intermédiaire | Source utilisée |
|---|---|
| `application_base.csv` | `application_train.csv` et `application_test.csv` |
| `bureau_features.csv` | `bureau.csv` et `bureau_balance.csv` |
| `previous_application_features.csv` | `previous_application.csv` |
| `pos_cash_features.csv` | `POS_CASH_balance.csv` |
| `installments_features.csv` | `installments_payments.csv` |
| `credit_card_features.csv` | `credit_card_balance.csv` |

Ces fichiers sont ensuite fusionnés pour construire le dataset final.

In [3]:
interim_files = [
    "application_base.csv",
    "bureau_features.csv",
    "previous_application_features.csv",
    "pos_cash_features.csv",
    "installments_features.csv",
    "credit_card_features.csv",
]

interim_summary = []

for file_name in interim_files:
    file_path = INTERIM_DATA_DIR / file_name

    if file_path.exists():
        df = pd.read_csv(file_path, nrows=5)
        full_shape = pd.read_csv(file_path, usecols=["SK_ID_CURR"]).shape

        interim_summary.append(
            {
                "file": file_name,
                "sample_rows": df.shape[0],
                "n_columns": df.shape[1],
                "n_rows": full_shape[0],
                "has_SK_ID_CURR": "SK_ID_CURR" in df.columns,
            }
        )
    else:
        interim_summary.append(
            {
                "file": file_name,
                "sample_rows": None,
                "n_columns": None,
                "n_rows": None,
                "has_SK_ID_CURR": False,
            }
        )

interim_summary_df = pd.DataFrame(interim_summary)
interim_summary_df

,file,sample_rows,n_columns,n_rows,has_SK_ID_CURR
0,application_base.csv,5,269,356251,True
1,bureau_features.csv,5,63,305811,True
2,previous_application_features.csv,5,207,338857,True
3,pos_cash_features.csv,5,23,337252,True
4,installments_features.csv,5,32,339587,True
5,credit_card_features.csv,5,110,103558,True


## Fusion des features

Les fichiers intermédiaires sont fusionnés avec la table principale à partir de la clé `SK_ID_CURR`.

Cette étape permet d’obtenir un dataset enrichi avec :
- les informations de la demande de crédit ;
- l’historique des anciens crédits ;
- les informations de paiement ;
- les données de carte de crédit ;
- les historiques POS/CASH.

La cible `TARGET` est conservée uniquement dans le jeu d’entraînement.

In [4]:
processed_files = [
    "train_processed.csv",
    "test_processed.csv",
    "train_modeling.csv",
    "test_modeling.csv",
]

processed_summary = []

for file_name in processed_files:
    file_path = PROCESSED_DATA_DIR / file_name

    if file_path.exists():
        df = pd.read_csv(file_path, nrows=1000)

        processed_summary.append(
            {
                "file": file_name,
                "n_columns": df.shape[1],
                "sample_rows_loaded": df.shape[0],
                "has_TARGET": "TARGET" in df.columns,
                "has_SK_ID_CURR": "SK_ID_CURR" in df.columns,
                "missing_rate_sample": df.isna().mean().mean(),
            }
        )
    else:
        processed_summary.append(
            {
                "file": file_name,
                "n_columns": None,
                "sample_rows_loaded": None,
                "has_TARGET": False,
                "has_SK_ID_CURR": False,
                "missing_rate_sample": None,
            }
        )

processed_summary_df = pd.DataFrame(processed_summary)
processed_summary_df

,file,n_columns,sample_rows_loaded,has_TARGET,has_SK_ID_CURR,missing_rate_sample
0,train_processed.csv,697,1000,True,True,0.202879
1,test_processed.csv,696,1000,False,True,0.176993
2,train_modeling.csv,658,1000,True,True,0.181036
3,test_modeling.csv,657,1000,False,True,0.153916


In [5]:
train_modeling_path = PROCESSED_DATA_DIR / "train_modeling.csv"
test_modeling_path = PROCESSED_DATA_DIR / "test_modeling.csv"

train_modeling = pd.read_csv(train_modeling_path)
test_modeling = pd.read_csv(test_modeling_path)

print("Train modeling shape :", train_modeling.shape)
print("Test modeling shape  :", test_modeling.shape)

train_modeling.head()

Train modeling shape : (307507, 658)
Test modeling shape  : (48744, 657)


,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,CC_SK_DPD_DEF_VAR,CC_NAME_CONTRACT_STATUS_ACTIVE_MEAN,CC_NAME_CONTRACT_STATUS_APPROVED_MEAN,CC_NAME_CONTRACT_STATUS_COMPLETED_MEAN,CC_NAME_CONTRACT_STATUS_DEMAND_MEAN,CC_NAME_CONTRACT_STATUS_REFUSED_MEAN,CC_NAME_CONTRACT_STATUS_SENT_PROPOSAL_MEAN,CC_NAME_CONTRACT_STATUS_SIGNED_MEAN,CC_NAME_CONTRACT_STATUS_NAN_MEAN,CC_COUNT
0,100002,1.0,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,-637.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0.0,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,-1188.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0.0,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,-225.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0.0,0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005,-3039.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0
4,100007,0.0,0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932,-3038.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
target_distribution = (
    train_modeling["TARGET"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
    .rename(columns={"index": "TARGET"})
)

target_distribution

,TARGET,proportion
0,0.0,0.91927
1,1.0,0.08073


## Nettoyage avant modélisation

Après fusion, certaines variables peuvent être peu exploitables.

Les principales opérations de nettoyage sont :
- suppression des colonnes avec trop de valeurs manquantes ;
- suppression des colonnes constantes ;
- conservation de `SK_ID_CURR` comme identifiant technique ;
- conservation de `TARGET` uniquement dans le jeu d'entraînement.

Le fichier `train_modeling.csv` est ensuite utilisé pour entraîner les modèles MLflow.

In [7]:
dropped_columns_path = PROJECT_ROOT / "reports" / "data_quality" / "modeling_dropped_columns.csv"

if dropped_columns_path.exists():
    dropped_columns = pd.read_csv(dropped_columns_path)
    dropped_columns
else:
    print("Le fichier modeling_dropped_columns.csv n'existe pas.")

## Conclusion de la préparation des features

La préparation des features a permis de transformer les différentes tables du dataset Home Credit en un dataset unique exploitable pour la modélisation.

Le dataset final d'entraînement contient une ligne par client et regroupe des informations issues de plusieurs sources :
- demande de crédit actuelle ;
- anciens crédits ;
- historique de paiement ;
- cartes de crédit ;
- crédits POS/CASH.

Cette étape est essentielle car les modèles de machine learning ne peuvent pas exploiter directement les tables relationnelles brutes.
Les agrégations permettent de résumer l’historique client sous forme de variables numériques.

Le fichier final utilisé pour la modélisation est `train_modeling.csv`.